## Getting and Formatting Historical Data

In this notebook, we collect historical data for options exchanges from "Deribit" via the help of *Tardis.dev* -- a tick-level historical and real-time cryptocurrency market data -- and formatting them appropriately so that we can these data later on our main option-trading algorithm.

"Deribit" makes the data for first day of each month available for free. This is the data that we are going to use.

- We focus on Bitcoin options, since they are the most liquid.
- We focus on at-the-money trades, since we have the most trading volume and a narrow bid-ask spread. This means reduce trading costs (although in our case we do not include trading costs). ATM options are also the most "dynamic" making them easier for hedging prices. For this reason we only include options that have $0.4 \leq \Delta \leq 0.6$.
- We only include options with less than a month expiry date. This, in combination with the fact that we get data only for the first day of each month, helps to avoid the same quote being traded on for consecutive months.

All these singificantly reduce the amount of quotes per month, making the overall program run faster.

In [ ]:
#Install Tardis --- CVS datasets for Trades (Uncomment the following line)
# !pip install tardis-dev

#Install cctx ctypto library --- CVS datasets for Trades (Uncomment the following line)
# !pip install ccxt

from tardis_dev import datasets, get_exchange_details
import ccxt
import logging
import asyncio
import nest_asyncio

import numpy as np
import pandas as pd
import dask.dataframe as dd
import os
import datetime as dt
import time

In [ ]:
#Log Event systems -- Comment if wanting to disable debug logs
logging.basicConfig(level=logging.DEBUG)
# Allow nested event loops
nest_asyncio.apply()

# Name of the files once the data is retrieved
def file_name(exchange, data_type, date, symbol, format):
    return f"{exchange}_{data_type}_{date.strftime('%Y-%m-%d')}_{symbol}.{format}.gz"


# returns data available from Deribit Trading Platform 
exchange_plt = "deribit"
deribit_details = get_exchange_details(exchange_plt)

# From_date list (1st day of each month)
# from_date_list = ["2020-01-01", "2020-02-01", "2020-03-01", "2020-04-01", "2020-05-01", "2020-06-01", "2020-07-01", "2020-08-01", "2020-09-01", "2020-10-01", "2020-11-01", "2020-12-01"]
# from_date_list = ["2023-01-01", "2023-02-01", "2023-03-01", "2023-04-01", "2023-05-01", "2023-06-01", "2023-07-01", "2023-08-01", "2023-09-01", "2023-10-01", "2023-11-01", "2023-12-01"]
from_date_list = ["2025-01-01", "2025-02-01", "2025-03-01", "2025-04-01", "2025-05-01", "2025-06-01", "2025-07-01", "2025-08-01", "2025-09-01", "2025-10-01", "2025-11-01", "2025-12-01"]

# To_date list (2nd day of each month)
# to_date_list = ["2020-01-02", "2020-02-02", "2020-03-02", "2020-04-02", "2020-05-02", "2020-06-02", "2020-07-02", "2020-08-02", "2020-09-02", "2020-10-02", "2020-11-02", "2020-12-02"]
# to_date_list = ["2023-01-02", "2023-02-02", "2023-03-02", "2023-04-02", "2023-05-02", "2023-06-02", "2023-07-02", "2023-08-02", "2023-09-02", "2023-10-02", "2023-11-02", "2023-12-02"]
to_date_list = ["2025-01-02", "2025-02-02", "2025-03-02", "2025-04-02", "2025-05-02", "2025-06-02", "2025-07-02", "2025-08-02", "2025-09-02", "2025-10-02", "2025-11-02", "2025-12-02"]
year = 2025#2020#2023

i = 0
for i in range(len(to_date_list)):
    datasets.download(
        # one of https://api.tardis.dev/v1/exchanges exchanges -- use 'id' value
        exchange = exchange_plt,
        # specify data types -- see available options https://docs.tardis.dev/downloadable-csv-files/data-types
        data_types = ["options_chain"],
        # change date ranges as needed
        from_date = from_date_list[i],
        to_date = to_date_list[i],
        # accepted values -- see https://docs.tardis.dev/historical-data-details/deribit
        symbols = ["OPTIONS"],
        # (optional) your API key to get access to non sample data as well
        api_key ="YOUR API KEY",
        # (optional) path where data will be downloaded into, default dir is './datasets'
        download_dir ="D:\Datasets_options",
        # (optional) - Customize file name/path - by default function 'file_name' is used
        get_filename = file_name
    )

### Change the variable *year* --- to collect data from that year.

Careful --- year 2019 starts from april --- If one wants information about 2019, they need to change the code.

Also recommend once the data is formatted --- delete the previous zip files.

In [ ]:
file_path_ind = []
year = 2025#2020#2023

for months in range(1,13):
    #month = str(months)
    month = f"{months:02d}"
    filename = "D:\Datasets_options\deribit_options_chain_{}-{}-01_OPTIONS.csv.gz".format(year,month)
    #"\Desktop\Datasets\deribit_options_chain_{}-{}-01_OPTIONS.csv.gz".format(year,month)
    file_path_ind.append(filename)
    # d = d + timedelta(months = 1)
print(file_path_ind)

The following cell can be used when the zip files are relatively medium size or one has enough RAM (e.g. with 16gb of RAM 2023-05-01 cannot be read using the read_csv file --- not enough memory).

For this reason the cell after the following one reads in 1_000_000 chunks instead, so that RAM usage can be reduced. 

In [ ]:
month_index =  ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]
month_num_index = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

for j in range(len(month_num_index)):
    print("========== Formating {}-{} ==========".format(month_index[j],year)) # or print(f"========== Recursion {month_index[j]} ==========")
    if os.path.exists(file_path_ind[j]):
        #read the files    
        start = time.time()
        deribit_df = pd.read_csv(file_path_ind[j], compression = "gzip", dtype=dtypes)
        end = time.time()
        print("It took", end - start, "seconds to read!")
        print("")
        # Contain only BTC and expire within 1 month
        BTC_df = deribit_df[deribit_df["symbol"].str.contains('BTC') & deribit_df["symbol"].str.contains(month_index[j])]
        # Delta Filter -- Consider only at-the-money trades
        atmBTC_df = BTC_df[BTC_df['delta'].between(0.5, 0.6) | BTC_df['delta'].between(-0.55, -0.45)]
        print("Length of ATMs with duplicates: ", len(atmBTC_df))
        print("")
        #Need to fix the timestamps -- Convert from ms to date format (see CSV schema https://docs.tardis.dev/downloadable-csv-files/data-types#options_chain)
        atmBTC_df["timestamp"] = pd.to_datetime(atmBTC_df["timestamp"],unit='us')
        atmBTC_df["local_timestamp"] = pd.to_datetime(atmBTC_df["local_timestamp"],unit='us')
        atmBTC_df["expiration"] = pd.to_datetime(atmBTC_df["expiration"],unit='us')
        # print(atmBTC_df["timestamp"].head())
        # print("")
        unique_price_movement_df = atmBTC_df.sort_values("timestamp").drop_duplicates(subset=['bid_price', 'ask_price'], keep='first')
        print(unique_price_movement_df["timestamp"].head())
        print("")
        print("Length of unique ATMs: ", len(unique_price_movement_df))
        print("")
    else:
        print(f"File not found: {file_path}")
        print("Please check the file path and try again.")
    
    #save file
    save_file ="D:\Datasets_options_formatted\Format_Dataset_{}-{}_OPTIONS.csv".format(year,month_num_index[j]) 
    unique_price_movement_df.to_csv(save_file, index=False)


This cell is used when the PC RAM is small or when the data is too large to open using csv.

In our case using 16Gb RAM, data for half the months of 2023 and almost all the months of 2025, needed the below treatement in order to read and then be formatted. 
Idea: just read in chunks and then concatinate, once each chunk has been formatted.

In [ ]:
chunk_size = 1_000_000 #500_000
month_index =  ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]
month_num_index = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

for j in range(len(month_num_index)):
    print("========== Formatting {}-{} ==========".format(month_index[j],year)) # or print(f"========== Recursion {month_index[j]} ==========")
    if os.path.exists(file_path_ind[j]):
        #read the files
        processed_chunks = []
        chunk_num = 0
        start = time.time()
        for chunk in pd.read_csv(file_path_ind[j], compression="gzip", chunksize=chunk_size):
            chunk_num +=1 
            print(f"This is Chunk no {chunk_num}:") 
            print("")
            print("Length of ATMs before formatting: ", len(chunk))
            # Contain only BTC and expire within 1 month
            BTC_df = chunk[chunk["symbol"].str.contains('BTC') & chunk["symbol"].str.contains(month_index[j])]
            # Delta Filter -- Consider only at-the-money trades
            atmBTC_df = BTC_df[BTC_df['delta'].between(0.5, 0.6) | BTC_df['delta'].between(-0.55, -0.45)]
            print("Length of ATMs with duplicates: ", len(atmBTC_df))
            print("")
            #Need to fix the timestamps -- Convert from ms to date format (see CSV schema https://docs.tardis.dev/downloadable-csv-files/data-types#options_chain)
            atmBTC_df["timestamp"] = pd.to_datetime(atmBTC_df["timestamp"],unit='us')
            atmBTC_df["local_timestamp"] = pd.to_datetime(atmBTC_df["local_timestamp"],unit='us')
            atmBTC_df["expiration"] = pd.to_datetime(atmBTC_df["expiration"],unit='us')
            # print(atmBTC_df["timestamp"].head())
            # print("")
            unique_price_movement_df = atmBTC_df.sort_values("timestamp").drop_duplicates(subset=['bid_price', 'ask_price'], keep='first')
            print(unique_price_movement_df["timestamp"].head())
            print("")
            print("Length of unique ATMs: ", len(unique_price_movement_df))
            print("")
            processed_chunks.append(unique_price_movement_df)
        end = time.time()          
        final_df = pd.concat(processed_chunks)# ignore_index=True)
        final_unique_df = final_df.sort_values("timestamp").drop_duplicates(subset=['bid_price', 'ask_price'], keep='first')
        print("Final Length of unique ATMs: ", len(final_unique_df))
        print("")
        print("It took", end - start, "seconds to read and format!")
        print("")  
    else:
        print(f"File not found: {file_path}")
        print("Please check the file path and try again.")
    
    #save file
    save_file ="D:\Datasets_options_formatted\Format_Dataset_{}-{}_OPTIONS.csv".format(year,month_num_index[j]) 
    final_unique_df.to_csv(save_file, index=False)